# PoC : Qwen3-8B + Activation Oracle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project/blob/main/notebooks/00_poc_qwen3_oracle.ipynb)

Objectif : vérifier que tout tourne de bout en bout, avec le code le plus simple possible.

1. **Qwen3-8B** répond à une question : le LLM cible fonctionne.
2. On lit ses **activations** à mi-profondeur (couche 18 sur 36) avec un hook PyTorch.
3. On donne ces activations à l'**Activation Oracle** et on lui pose une question dessus.

L'oracle ([Karvonen et al., 2025](https://arxiv.org/abs/2512.15674)) est un **adaptateur LoRA entraîné
sur Qwen3-8B lui-même**. On charge donc le modèle une seule fois :
LoRA désactivé = Qwen3-8B d'origine, LoRA activé = oracle.
Code de référence : [adamkarvonen/activation_oracles](https://github.com/adamkarvonen/activation_oracles)
(notebook `experiments/activation_oracle_demo.ipynb`).

**Sur Colab** : *Exécution → Modifier le type d'exécution → GPU*. Le T4 gratuit suffit (le modèle est
alors chargé en 8 bits). Premier lancement : ~5 min pour télécharger Qwen3-8B (16 Go).

## 0. Setup

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project.git"
BRANCH = "main"  # mettre ici sa branche pour tester du code pas encore mergé

if IN_COLAB:
    # Clone le dépôt (ou le met à jour), puis installe le package procom et ses dépendances
    !git clone -q {REPO_URL} /content/procom 2>/dev/null
    !cd /content/procom && git fetch -q && git checkout -q {BRANCH} && git pull -q
    !pip install -q -e /content/procom
    sys.path.insert(0, "/content/procom/src")  # procom importable sans redémarrer le runtime

In [ ]:
import torch

import procom  # noqa: F401  (vérifie que le package du dépôt est bien installé)

assert torch.cuda.is_available(), "Pas de GPU : Exécution → Modifier le type d'exécution → GPU"
print(torch.cuda.get_device_name(0))

## 1. Chargement de Qwen3-8B et de l'oracle

Qwen3-8B en bf16 pèse ~16 Go : ça ne rentre pas sur un T4 (15 Go), on le charge alors en 8 bits.

In [ ]:
import torch.nn.functional as F
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-8B"
ORACLE_LORA = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
load_in_8bit = gpu_memory_gb < 20
quantization = BitsAndBytesConfig(load_in_8bit=True) if load_in_8bit else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto", quantization_config=quantization
)
device = model.device
layers = model.model.layers  # les blocs transformer, où l'on lit et injecte les activations

model = PeftModel.from_pretrained(model, ORACLE_LORA)  # ajoute la LoRA de l'oracle
model.eval()
torch.set_grad_enabled(False)

print(f"GPU {gpu_memory_gb:.0f} Go | 8 bits : {load_in_8bit} | {len(layers)} couches")

## 2. Qwen3-8B seul

Dans `model.disable_adapter()`, la LoRA est coupée : c'est le Qwen3-8B d'origine.

In [ ]:
def chat(messages, max_new_tokens=60):
    """Fait répondre Qwen3-8B (sans l'oracle) à une conversation."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
    with model.disable_adapter():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0, inputs.input_ids.shape[1] :], skip_special_tokens=True)


print(chat([{"role": "user", "content": "What is the capital of France? Answer in one sentence."}]))

## 3. Lire les activations de Qwen3-8B

Un *forward hook* PyTorch récupère la sortie d'une couche (le flux résiduel) pendant un forward.
L'oracle a été entraîné sur des activations prises à 25, 50 et 75 % de profondeur ; comme dans
l'article, on prend 50 %, soit la couche 18 de Qwen3-8B.

In [ ]:
ACT_LAYER = len(layers) // 2  # 50 % de profondeur : couche 18 pour Qwen3-8B


def get_activations(messages, layer=ACT_LAYER):
    """Activations de Qwen3-8B (sans LoRA) en sortie de `layer`, une ligne par token."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
    captured = {}

    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["acts"] = hidden[0].detach()  # (nb_tokens, d_model)

    handle = layers[layer].register_forward_hook(hook)
    try:
        with model.disable_adapter():
            model(**inputs)
    finally:
        handle.remove()  # toujours retirer le hook, sinon il reste actif pour la suite
    tokens = [tokenizer.decode(t) for t in inputs.input_ids[0]]
    return captured["acts"], tokens

In [ ]:
target = [
    {
        "role": "user",
        "content": "The philosopher who drank hemlock taught a student who founded an academy. "
        "That student's most famous pupil was",
    }
]
acts, tokens = get_activations(target)
print("activations :", tuple(acts.shape))
print(list(enumerate(tokens)))

## 4. Interroger l'oracle

Le prompt de l'oracle a le format utilisé à l'entraînement :

```
Layer: 18
 ? ? ? … ?
<question>
```

avec un token ` ?` par vecteur d'activation. Pendant le forward de l'oracle, un hook en sortie de la
**couche 1** ajoute chaque vecteur `v` sur son ` ?`, remis à l'échelle de la norme du résiduel `h` à
cette position : `h ← h + ‖h‖ · v / ‖v‖`.

In [ ]:
INJECT_LAYER = 1  # l'oracle a été entraîné avec une injection en sortie de la couche 1
PLACEHOLDER = " ?"


def ask_oracle(acts, question, max_new_tokens=50):
    """Pose `question` à l'oracle sur les vecteurs d'activation `acts` (nb_vecteurs, d_model)."""
    n = len(acts)
    prompt = f"Layer: {ACT_LAYER}\n" + PLACEHOLDER * n + " \n" + question
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)

    # Positions des n tokens " ?" dans le prompt (les n premiers, la question peut en contenir)
    placeholder_id = tokenizer.encode(PLACEHOLDER, add_special_tokens=False)[0]
    positions = (inputs.input_ids[0] == placeholder_id).nonzero().flatten()[:n]
    assert len(positions) == n and positions[-1] - positions[0] == n - 1

    def inject(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.shape[1] == 1:  # étapes de génération (1 token à la fois) : rien à injecter
            return
        h = hidden[0, positions]
        v = F.normalize(acts, dim=-1).to(h.dtype)
        hidden[0, positions] = h + h.norm(dim=-1, keepdim=True) * v

    handle = layers[INJECT_LAYER].register_forward_hook(inject)
    try:
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    finally:
        handle.remove()
    return tokenizer.decode(output[0, inputs.input_ids.shape[1] :], skip_special_tokens=True)

On pose la même question avec deux entrées : toutes les activations du prompt, puis le seul vecteur
du token ` pupil`. Sur cet exemple, la démo des auteurs obtient, pour le token ` pupil` :
*« The assistant is currently thinking about Socrates, Plato, and Aristotle. »*

In [ ]:
question = "Can you name which person the model is thinking about?"
pupil = tokens.index(" pupil")

print("Toutes les positions :", ask_oracle(acts, question))
print("Token ' pupil' seul  :", ask_oracle(acts[pupil : pupil + 1], question))

## Pistes pour la suite

- Changer `target` et `question` (ex. *« What is the topic of this text? »*,
  *« Answer with a single word. What emotion is being felt here? »*).
- Contrôle : passer des activations aléatoires (`torch.randn_like(acts)`) pour voir ce que répond
  l'oracle quand il n'y a aucune information.
- Premier pas vers le projet : mettre à zéro une partie de `acts` avant `ask_oracle` et comparer.